<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/TransformerTranslate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input,Dense,Embedding,Layer
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
subjects = ["I", "You", "He", "She", "We", "They"]
verbs = ["eat", "like", "see", "have", "want", "buy", "find", "need"]
objects = ["apple", "car", "book", "coffee", "house", "dog", "cat", "food"]

# English → Spanish mapping (simple)
translation = {
    "I": "Yo", "You": "Tú", "He": "Él", "She": "Ella", "We": "Nosotros", "They": "Ellos",
    "eat": "como", "like": "gusta", "see": "veo", "have": "tengo",
    "want": "quiero", "buy": "compro", "find": "encuentro", "need": "necesito",
    "apple": "manzana", "car": "coche", "book": "libro", "coffee": "café",
    "house": "casa", "dog": "perro", "cat": "gato", "food": "comida"
}

input_texts = []
target_texts = []

for s in subjects:
    for v in verbs:
        for o in objects:
            eng = f"{s} {v} {o}"
            spa = f"{translation[s]} {translation[v]} {translation[o]}"
            input_texts.append(eng)
            target_texts.append(spa)

print(len(input_texts))  # 6 * 8 * 8 = 384

384


In [ ]:
extra_phrases = [
    ("Good morning", "Buenos días"),
    ("Thank you", "Gracias"),
    ("I am happy", "Estoy feliz"),
    ("Where is the car", "Dónde está el coche"),
    ("I need help", "Necesito ayuda"),
    ("See you soon", "Hasta pronto"),
    ("I am learning", "Estoy aprendiendo"),
    ("The sky is blue", "El cielo es azul"),
    ("The water is cold", "El agua está fría"),
    ("It is hot today", "Hace calor hoy")
]

for eng, spa in extra_phrases:
    input_texts.append(eng)
    target_texts.append(spa)

In [ ]:
#Add startseq and endseq
target_texts = ['startseq ' + t + ' endseq' for t in target_texts]

In [ ]:
#Tokenization
input_token = Tokenizer()
input_token.fit_on_texts(input_texts)
input_sequences = input_token.texts_to_sequences(input_texts)

output_token = Tokenizer()
output_token.fit_on_texts(target_texts)
output_sequences = output_token.texts_to_sequences(target_texts)

In [ ]:
#Get number of unique words
input_vocab_length = len(input_token.word_index) + 1
output_vocab_length = len(output_token.word_index) + 1

In [ ]:
input_vocab_length

41

In [ ]:
max_input_length = max(len(seq) for seq in input_sequences)

In [ ]:
max_input_length

4

In [ ]:
max_output_length = max(len(seq) for seq in output_sequences)

In [ ]:
input_sequences = pad_sequences(input_sequences,maxlen=max_input_length,padding='post')
output_sequences = pad_sequences(output_sequences,maxlen=max_output_length,padding='post')

In [ ]:
decoder_input = output_sequences[:,:-1]
decoder_output = output_sequences[:,1:]

docoder_output = np.expand_dims(decoder_output,-1)

In [31]:
#Attention class
class Attention(Layer):
  def __init__(self,d_model):
      super().__init__()
      self.wq = Dense(d_model)
      self.wk = Dense(d_model)
      self.wv = Dense(d_model)

  def call(self,q,k,v):
    Q = self.wq(q)
    K = self.wk(k)
    V = self.wv(v)

    scores = tf.matmul(Q,K,transpose_b=True)
    dk = tf.cast(tf.shape(K)[-1],tf.float32)
    scores = scores / tf.sqrt(dk) # we do the square root so the value become more shorter, otherwise the model gives priority to bigger values.

    weights = tf.nn.softmax(scores)
    output = tf.matmul(weights,V)

    return output

In [35]:
#self attention
encoder_input = Input(shape=(max_input_length,))
x = Embedding(input_vocab_length,70)(encoder_input)

encoder_output = Attention(70)(x,x,x)

decoder_inputs = Input(shape=(max_output_length - 1,))
y = Embedding(output_vocab_length,80)(decoder_inputs)

decoder_outputs = Attention(70)(y,y,y)

final_decoder = Attention(70)(decoder_outputs,encoder_output,encoder_output) # here the idea is when our model goes in training, input we need for Q is spanish and for K and V we need english self attention output

output = Dense(output_vocab_length,activation='softmax')(final_decoder)

model = Model([encoder_input,decoder_inputs],output)

model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

model.fit([input_sequences,decoder_input],docoder_output,epochs=150,validation_split=0.2)

Epoch 1/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 77ms/step - accuracy: 0.1663 - loss: 3.7409 - val_accuracy: 0.2000 - val_loss: 3.6866
Epoch 2/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.2000 - loss: 3.4469 - val_accuracy: 0.2000 - val_loss: 3.4184
Epoch 3/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.2000 - loss: 2.9881 - val_accuracy: 0.2000 - val_loss: 3.3815
Epoch 4/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.2000 - loss: 2.8467 - val_accuracy: 0.2152 - val_loss: 3.4157
Epoch 5/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.2000 - loss: 2.7676 - val_accuracy: 0.2000 - val_loss: 3.4980
Epoch 6/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2000 - loss: 2.7207 - val_accuracy: 0.2101 - val_loss: 3.5873
Epoch 7/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2000 - loss: 2.6856 - val_accuracy: 0.2051 - val_loss: 3.6550
Epoch 8/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2000 - loss: 2.6466 - val_accuracy: 0.